## Imports

In [1]:
# SimpleDirectoryReader is dynamic, detects file type and uses appropriate reader
from llama_index.core import SimpleDirectoryReader, Document, VectorStoreIndex, Settings, PromptTemplate, StorageContext
from llama_index.core.utilities.sql_wrapper import SQLDatabase
from llama_index.core.query_engine import NLSQLTableQueryEngine, KnowledgeGraphQueryEngine
from llama_index.core.workflow import Workflow, StartEvent, StopEvent, step, Context, Event
from llama_index.core.retrievers import SQLRetriever
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.openai import OpenAI
from llama_parse import LlamaParse

from transformers import AutoTokenizer

import pandas as pd, re, ast, textwrap
from sqlalchemy import create_engine

import openai
from neo4j import GraphDatabase
from llama_index.graph_stores.neo4j import Neo4jGraphStore

from dotenv import load_dotenv
import torch
import os
import re
import json

load_dotenv()
hf_token = os.getenv("HUGGINGFACE_TOKEN")
data_directory = os.getenv("VECTOR_DATASET_DIR")
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# Graph database connection
uri = "neo4j://127.0.0.1:7687"
username = "neo4j"
password = os.getenv("NEO4J_PASSWORD")
driver = GraphDatabase.driver(uri, auth=(username, password))

# Verify the connection
try:
    driver.verify_connectivity()
    print("Connection to local Neo4j Desktop database successful!")
except Exception as e:
    print(f"Failed to connect to Neo4j Desktop. Please ensure the database is running. Error: {e}")

Connection to local Neo4j Desktop database successful!


## Vector/Graph Store Data Ingestion

In [2]:
# Check if directory exists
if not data_directory or not os.path.isdir(data_directory):
    raise ValueError(
        f"The path '{data_directory}' is not a valid directory. "
        "Please check that the VECTOR_DATASET_DIR variable is set correctly in your .env file "
        "and that the directory actually exists."
    )

# Initialize LlamaParse with your API key
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
if not llama_cloud_api_key:
    raise ValueError("LLAMA_CLOUD_API_KEY not found in your .env file. Please get a key from https://cloud.llamaindex.ai")

parser = LlamaParse(
    api_key=llama_cloud_api_key,
    result_type="markdown",
    verbose=True
)

# Separate the file paths based on their type (PDF vs. other)
pdf_filepaths = []
other_filepaths = []
for filename in os.listdir(data_directory):
    file_path = os.path.join(data_directory, filename)
    if os.path.isfile(file_path):
        if filename.lower().endswith('.pdf'):
            pdf_filepaths.append(file_path)
        else:
            other_filepaths.append(file_path)

print(f"--- Found {len(pdf_filepaths)} PDF(s) and {len(other_filepaths)} other file(s) to process. ---")

# Process the files in batches
all_documents = []

# Process all PDFs in a single batch call to LlamaParse
if pdf_filepaths:
    print("\n- Parsing PDF files with LlamaParse...")
    try:
        # Calling parser.load_data() with a LIST of files is the correct way
        pdf_docs = parser.load_data(pdf_filepaths)
        all_documents.extend(pdf_docs)
        print(f"  -> Successfully parsed {len(pdf_filepaths)} PDF file(s).")
    except Exception as e:
        print(f"  -> FAILED to parse PDFs with LlamaParse. Error: {e}")

# Process all other files in a single batch call to SimpleDirectoryReader
if other_filepaths:
    print("\n- Parsing other files with SimpleDirectoryReader...")
    try:
        other_docs = SimpleDirectoryReader(input_files=other_filepaths).load_data()
        all_documents.extend(other_docs)
        print(f"  -> Successfully parsed {len(other_filepaths)} other file(s).")
    except Exception as e:
        print(f"  -> FAILED to parse other files. Error: {e}")

# The 'documents' variable should now contain all chunks from all parsed files
documents = all_documents
print(f"\n--- Ingestion complete ---")
print(f"Successfully loaded and chunked a total of {len(documents)} document(s) from all files in '{data_directory}'.")


--- Found 1 PDF(s) and 0 other file(s) to process. ---

- Parsing PDF files with LlamaParse...


Parsing files:   0%|          | 0/1 [00:00<?, ?it/s]

Started parsing the file under job_id d286d35e-9e5b-4ff6-82fc-0ba2f1e4ba0d


Parsing files: 100%|██████████| 1/1 [00:16<00:00, 16.67s/it]

  -> Successfully parsed 1 PDF file(s).

--- Ingestion complete ---
Successfully loaded and chunked a total of 2 document(s) from all files in 'Graph_Dataset'.


## SQL Datastore Ingestion

In [ ]:
# Create an in-memory SQLite database
# This database exists only as long as the script is running
engine = create_engine("sqlite:///:memory:")

# --- Dynamically load all CLEANED CSVs from the 'SQL_Dataset' directory ---
# This should point to the folder where your 'run_SQL_cleaning.py' script saved the files.
sql_data_directory = "SQL_Dataset" 
table_names = [] # To keep track of the tables we create

print(f"Searching for cleaned CSV files to ingest in '{sql_data_directory}'...")

# Check if the directory exists to avoid errors
if not os.path.isdir(sql_data_directory):
    print(f"Error: The directory '{sql_data_directory}' was not found. Please ensure the cleaning script ran successfully.")
else:
    for filename in os.listdir(sql_data_directory):
        if filename.endswith(".csv"):
            try:
                file_path = os.path.join(sql_data_directory, filename)
                
                # 1. Load the already-cleaned CSV into a DataFrame
                cleaned_df = pd.read_csv(file_path)
                
                # 2. Create a clean table name from the filename
                # Example: "cleaned_m891481.csv" -> "cleaned_m891481"
                table_name = os.path.splitext(filename)[0].lower()
                table_names.append(table_name)
                
                # 3. Ingest the cleaned DataFrame into the SQL database
                cleaned_df.to_sql(table_name, engine, index=False, if_exists='replace')
                
                print(f" - Successfully ingested '{filename}' into SQL table '{table_name}'")
            except Exception as e:
                print(f" - FAILED to ingest {filename}. Error: {e}")

print(f"\nIn-memory SQL database created and populated with {len(table_names)} table(s).")
sql_database = SQLDatabase(engine)


## Llama 3.1 8B Instruct

In [3]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Initialize the tokenizer to get the token ID for our stop sequence
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
# The semicolon is our desired stop character. Get its token ID.
semicolon_token_id = tokenizer.convert_tokens_to_ids(";")

# Now, initialize the LLM with the correct stop condition
llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    device_map="auto",
    model_kwargs={"token": hf_token, "torch_dtype": torch.bfloat16},
    # Use 'eos_token_id' which is the correct parameter for this purpose
    generate_kwargs={
        "temperature": 0.1,
        "do_sample": True,
        # This tells the model to stop generating as soon as it outputs a semicolon
        "eos_token_id": semicolon_token_id,
    }
)

print("HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.


## OpenAI gpt model (Entity & Relationship Extraction for Graph datastore)

In [8]:
openai_llm = OpenAI(
    api_key=openai_api_key,
    model="gpt-4o",
    temperature=0.0,
    # Uncomment to set a timeout
    # timeout=120.0,
)

print("OpenAI LLM initialized successfully.")

OpenAI LLM initialized successfully.


## Vector Embeddings

In [ ]:
Settings.llm = llm
Settings.embed_model = "local:BAAI/bge-small-en-v1.5"

print("Global settings configured with Llama 3.1 and bge-small embedding model.")

## Indexing

In [ ]:
# Pass the list 'documents' directly, creating separate index entries for each document chunk
# By default, VectorStoreIndex chunk size is set to 1024 characters
# And chunk overlap is set to 20% of the chunk size
# This means each chunk will be 1024 characters long, with a 205 character overlap
# This is suitable for most text data, but can be adjusted if needed
index = VectorStoreIndex.from_documents(
    documents
)

print(f"Vector store index has been built successfully from {len(documents)} source document(s).")


## Entity & Relationship Extraction with gpt

In [16]:
extraction_prompt_str = """
You are an expert data architect building a knowledge graph from unstructured text.
Your task is to extract factual relationship triplets, assigning a type to each entity and creating a canonical relationship name.

INSTRUCTIONS:
1.  Identify the key entities (subjects and objects) in the text.
2.  For each entity, assign it a generic type (Node Label) from this list:
    ['Organization', 'Person', 'Project', 'Technology', 'Location', 'Event', 'Miscellaneous']
3.  For each relationship (Predicate), create a short, descriptive, active verb phrase in ALL_CAPS_SNAKE_CASE.
    - GOOD: 'HAS_PARTNER', 'VISITED', 'ANNOUNCED', 'LED_BY'
    - BAD: 'was the primary reason for', 'signed an agreement with'
4.  The final output MUST be a valid JSON list of objects. Each object must contain these five keys: "subject", "predicate", "object", "subject_type", "object_type".
5.  Your response must contain ONLY the JSON list and nothing else.

TEXT_TO_ANALYZE:
{text_chunk}
---

JSON_OUTPUT:
"""

extraction_prompt_tmpl = PromptTemplate(extraction_prompt_str)

# structuring_prompt_str = """
# You are an expert data architect. You are given a flat list of relationship triplets.
# Your task is to organize them into a meaningful, nested JSON hierarchy, starting with 'HTX' as the root.
# - The keys should represent entities.
# - The values can be a nested object or a list of entities.
# - Use common verb phrases as keys to represent relationships where appropriate (e.g., "INCLUDES", "HAS_PARTNER").
# The final output MUST be a single, clean JSON object.
# ---
# FLAT LIST OF TRIPLETS:
# {triplets_list}
# ---
# HIERARCHICAL JSON OUTPUT:
# """

# structuring_prompt_tmpl = PromptTemplate(structuring_prompt_str)

## Ingestion into Neo4j function

In [17]:
def parse_llm_json_response(text: str) -> list:
    """Finds and safely parses a JSON list from a string."""
    # This regex is robust enough to find a JSON list within surrounding text.
    match = re.search(r'\[\s*\{.*\}\s*\]', text, re.DOTALL)
    if not match:
        print("-> Warning: No valid JSON list found in the LLM's response.")
        return []
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        print("-> Warning: The LLM response contained a malformed JSON list.")
        return []

def ingest_dynamic_triplets(driver, triplets: list[dict]):
    """
    Ingests a list of dynamically generated triplets into Neo4j,
    using the types and predicates provided by the LLM.
    """
    nodes_created = 0
    rels_created = 0
    with driver.session() as session:
        for triplet in triplets:
            # Validate that the triplet has all the necessary keys
            if not all(k in triplet for k in ["subject", "predicate", "object", "subject_type", "object_type"]):
                continue

            # Clean up the labels and relationship type to be valid Cypher identifiers
            subject_label = re.sub(r'\W+', '', triplet['subject_type'])
            object_label = re.sub(r'\W+', '', triplet['object_type'])
            predicate = re.sub(r'\W+', '', triplet['predicate'].upper())

            # MERGE nodes dynamically using their assigned labels
            session.run(f"""
                MERGE (s:{subject_label} {{name: $subject}})
                MERGE (o:{object_label} {{name: $object}})
            """, subject=triplet['subject'], object=triplet['object'])
            nodes_created += 2

            # MERGE the relationship dynamically using the generated predicate
            session.run(f"""
                MATCH (s:{subject_label} {{name: $subject}})
                MATCH (o:{object_label} {{name: $object}})
                MERGE (s)-[r:`{predicate}`]->(o)
            """, subject=triplet['subject'], object=triplet['object'])
            rels_created += 1

    print(f"-> Success: Ingested/merged {nodes_created} nodes and {rels_created} relationships from this chunk.")

## Process and Ingest Data into Neo4j

In [18]:
# Re-open the driver and clear the database before ingestion
driver = GraphDatabase.driver(uri, auth=(username, password))
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("Database cleared.")

print(f"\n--- Processing {len(documents)} document(s) with Guided OIE pipeline... ---")
for doc in documents:
    doc_filename = doc.metadata.get('file_name', 'Unknown Document')
    print(f"Processing document: {doc_filename}")

    text_chunks = textwrap.wrap(doc.text, width=4000, break_long_words=False)

    for i, chunk in enumerate(text_chunks):
        print(f"  - Processing chunk {i+1}/{len(text_chunks)}...")

        # SINGLE LLM CALL: Use the new guided prompt. No schema needed.
        response_str = openai_llm.predict(
            extraction_prompt_tmpl,
            text_chunk=chunk
        )
        
        # Parse the response to get a clean list of triplets
        parsed_triplets = parse_llm_json_response(response_str)
        
        # Ingest the dynamically generated triplets into Neo4j
        if parsed_triplets:
            ingest_dynamic_triplets(driver, parsed_triplets)

print(f"\n--- Knowledge graph ingestion complete ---")
driver.close()


Database cleared.

--- Processing 2 document(s) with Guided OIE pipeline... ---
Processing document: Unknown Document
  - Processing chunk 1/1...
-> Success: Ingested/merged 28 nodes and 14 relationships from this chunk.
Processing document: Unknown Document
  - Processing chunk 1/1...
-> Success: Ingested/merged 24 nodes and 12 relationships from this chunk.

--- Knowledge graph ingestion complete ---


## Vector Query

In [ ]:
# Define query
qa_prompt_template_str = (
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "You are a helpful assistant. Based on the context information provided, "
    "answer the user's query in a clear and conversational manner. "
    "Do not just copy the context. Synthesize the information to respond directly to the question.\n"
    "Query: {query_str}\n"
    "Answer: "
)
qa_prompt_template = PromptTemplate(qa_prompt_template_str)


query_engine = index.as_query_engine(
    text_qa_template=qa_prompt_template,
    response_mode="compact",  # response_Mode controls how LlamaIndex uses retrieved chunks to generate answer
    similarity_top_k=3       # Retrieve the top 3 most relevant chunks
)

print("Query engine is ready with advanced synthesis settings.")


## Generate Summary of each SQL Table

In [ ]:
summary_prompt_str = """\
Provide a short, one-sentence summary for a table that has the following columns.
Your response MUST be ONLY the summary text and nothing else.

Columns:
{table_columns}

Summary: """
summary_prompt_tmpl = PromptTemplate(summary_prompt_str)

table_summaries = {}

print("--- Starting one-time summary generation ---")
print("This may take some time and consume a lot of memory.")

for table_name in table_names:
    try:
        print(f"  - Generating summary for: {table_name}")
        # Get ONLY the column names to save memory
        df = pd.read_sql(f"SELECT * FROM {table_name} LIMIT 1", engine)
        table_columns = str(df.columns.tolist())

        # Generate the summary using your local LLM
        summary = llm.predict(summary_prompt_tmpl, table_columns=table_columns).strip()
        table_summaries[table_name] = summary
        print(f"    -> Success.")

    except Exception as e:
        print(f"    -> FAILED to generate summary for {table_name}. Assigning default. Error: {e}")
        table_summaries[table_name] = "No summary available for this table."

# Save the generated summaries to a file
summary_file_path = "table_summaries.json"
with open(summary_file_path, 'w') as f:
    json.dump(table_summaries, f, indent=4)

print(f"\n--- Summaries saved to '{summary_file_path}' ---")
print(json.dumps(table_summaries, indent=4))

## Context for Text-To-SQL LLM

In [ ]:
schema_parts = []
summary_file_path = "table_summaries.json"

print(f"--- Loading pre-computed summaries from '{summary_file_path}' ---")
with open(summary_file_path, 'r') as f:
    table_summaries = json.load(f)

# Build the context string using the loaded summaries
for table_name in table_names:
    summary = table_summaries.get(table_name, "No summary available.")
    raw_schema = sql_database.get_single_table_info(table_name)
    schema_parts.append(
        f"Table Name: {table_name}\n"
        f"Table Summary: {summary}\n"
        f"Table Schema: {raw_schema}"
    )

schema_info = "\n\n".join(schema_parts)
print("\n--- Final context being sent to Text-to-SQL LLM ---")
print(schema_info)

## Raw SQL Query

In [ ]:
# Define user query
query_text_sql = "What was the total number of robbery cases in the Central Police Division in 2023?"

text_to_sql_prompt_template_str = (
        "You are an expert SQL generator. Analyze the user's question and the provided database context to generate a single, syntactically correct SQLite query.\n\n"
        "### INSTRUCTIONS\n"
        "1. Examine the table summaries to understand what data is in each table.\n"
        "2. IMPORTANT: If the user's question can be answered using a single table, you MUST use only that table. Do not create unnecessary JOINs.\n"
        "3. For string comparisons in WHERE clauses, use the `LOWER()` function on both the column and the value to ensure case-insensitive matching.\n"
        "4. Your response MUST be ONLY the single, raw SQL query.\n\n"
        "### DATABASE CONTEXT\n{schema}\n\n"
        "### QUESTION\n{query_str}\n\n"
        "### SQL QUERY\n"
    )
text_to_sql_prompt = PromptTemplate(text_to_sql_prompt_template_str)
    
# Generate the query using the main LLM
raw_sql_response = llm.predict(text_to_sql_prompt, schema=schema_info, query_str=query_text_sql)

## Clean SQL Query

In [ ]:
# Define the cleaning function
def extract_first_sql_query(raw_text: str) -> str:
    match = re.search(r"SELECT\s.*?;", raw_text, flags=re.DOTALL | re.IGNORECASE)
    return match.group(0).strip() if match else ""
    
clean_sql_query = extract_first_sql_query(raw_sql_response)
print(f"\n--- Cleaned SQL to be executed ---\n{clean_sql_query}")

## Synthesize SQL Prompt

In [ ]:
response_from_db = sql_database.run_sql(clean_sql_query)

period_token_id = tokenizer.convert_tokens_to_ids(".")

synthesis_prompt_str = synthesis_prompt_str = (
        "Based on the following data, answer the user's question in a single, complete sentence.\n\n"
        "User's Question: {original_question}\n"
        "Data from Database: {sql_result}\n\n"
        "Answer: "
    )
final_response = llm.predict(
    PromptTemplate(synthesis_prompt_str), 
    original_question=query_text_sql, 
    sql_result=str(response_from_db),
    eos_token_id = period_token_id)

clean_final_response = final_response.strip().split('.')[0] + '.'

## Vector Example Usage

In [ ]:
query_text_vector = "What is the total number of physical crime cases in 2023?"
vector_response = query_engine.query(query_text_vector)
print(str(vector_response))

## SQL Example Usage

In [ ]:
print(f"--- Running query: \"{query_text_sql}\" ---")
 
try:
    print("\n" + "="*20 + " FINAL ANSWER " + "="*20)
    print(textwrap.fill(clean_final_response, 80))

except Exception as e:
    print(f"\nAn error occurred during the manual workflow: {e}")


## Graph Query Engine

In [ ]:
graph_store = Neo4jGraphStore(username, password, uri)

storage_context = StorageContext.from_defaults(graph_store=graph_store)

# Prompt template to generate cypher query
DEFAULT_KG_QUERY_SYNTHESIS_TMPL = (
"You are an expert Cypher query generator. Your sole task is to generate a single, "
"syntactically correct Cypher query to answer the user's question based on the provided graph schema. "
"Do not provide any explanations, introductory text, or markdown formatting. "
"Your response MUST be ONLY the raw Cypher query and nothing else. "
"Start your response directly with a Cypher keyword like 'MATCH' or 'OPTIONAL MATCH'.\n\n"
"Schema:\n"
"---------------------\n"
"{schema}\n"
"---------------------\n"
"User's Question: {query_str}\n"
"Cypher Query:"
)

# Prompt template to generate response
DEFAULT_RESPONSE_SYNTHESIS_TMPL = (
    "You are a helpful assistant. You have been provided with the results of a Cypher query "
    "from a knowledge graph and the original user question. "
    "Synthesize a conversational answer based on the provided information. "
    "Do not mention Cypher or the knowledge graph in your response.\n"
    "User question: {query_str}\n"
    "Query results: {context_str}\n"
    "Answer: "
)

kg_query_synthesis_prompt = PromptTemplate(DEFAULT_KG_QUERY_SYNTHESIS_TMPL)
kg_response_answer_prompt = PromptTemplate(DEFAULT_RESPONSE_SYNTHESIS_TMPL)


# Initialize the query engine with the storage_context
graph_query_engine = KnowledgeGraphQueryEngine(
    storage_context=storage_context,
    graph_query_synthesis_prompt=kg_query_synthesis_prompt,
    graph_response_answer_prompt=kg_response_answer_prompt,
    verbose=True # Check Cypher Query
)

print("Knowledge graph query engine is ready.")

## Extract Cypher Text to Query Datastore

## Query Graph Datastore

In [ ]:
# Define question for graph
query_text_graph = "How much was lost to job scam?"

# Query the engine
graph_response = graph_query_engine.query(query_text_graph)

# Print the response
print(str(graph_response))